# Session 3 — Project Tasks — P4A 2026
**Group 23**

In [ ]:
import os
import pandas as pd
import numpy as np
import sqlalchemy as sa
from dotenv import load_dotenv

load_dotenv()

DB_HOST = "3de0dac0-8513-4220-9ee7-414dc040c138.bn2a2uid0up8mv7mv2ig.databases.appdomain.cloud"
DB_PORT = "31131"
DB_NAME = "linkedin_jobs"

url = sa.engine.URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST", DB_HOST),
    port=int(os.getenv("DB_PORT", DB_PORT)),
    database=os.getenv("DB_NAME", DB_NAME),
)

engine = sa.create_engine(url, connect_args={"ssl": {"check_hostname": False}})
print(url)

## Task 1 — Load the staging table

In [ ]:
df_postings = pd.read_sql("SELECT * FROM a20254350.postings_with_benefits", engine)

print(f"Shape: {df_postings.shape}")
print(f"Benefits column present: {'benefits' in df_postings.columns}")
df_postings.head()

## Task 2 — Explore the benefits column

In [ ]:
pct_with_benefits = df_postings['benefits'].notna().mean() * 100
print(f"{pct_with_benefits:.1f}% of postings have at least one benefit listed")

In [ ]:
top10_benefits = (
    df_postings['benefits']
    .dropna()
    .str.split(';')
    .explode()
    .str.strip()
    .value_counts()
    .head(10)
)
print("Top 10 most common benefits:")
print(top10_benefits)

**Insight:** Medical insurance, 401K and dental insurance dominate LinkedIn job postings, suggesting that US-based companies heavily compete on healthcare and retirement benefits to attract candidates.

## Task 3 — Load the companies data

In [ ]:
df_companies = pd.read_sql("SELECT * FROM linkedin_jobs.companies_companies", engine)

print(f"Shape: {df_companies.shape}")
df_companies.head()

## Task 4 — Assess data quality

In [ ]:
for col in ['company_size', 'country', 'city']:
    pct_null = df_companies[col].isna().mean() * 100
    print(f"{col}: {pct_null:.1f}% null")

**Comment:** High null rates in `company_size` limit its usefulness for size-based segmentation. Missing `country` and `city` values reduce geographic analysis reliability — any location-based conclusions should account for this coverage gap.

## Task 5 — Filter to US companies

In [ ]:
df_us = df_companies[df_companies['country'] == 'United States'].copy()
print(f"US companies: {len(df_us)}")

## Task 6 — Count companies by size

In [ ]:
size_counts = df_us['company_size'].value_counts().sort_index()
print("Companies per size (1=smallest, 7=largest):")
print(size_counts)
print(f"\nMost common size:  {size_counts.idxmax()}")
print(f"Least common size: {size_counts.idxmin()}")

## Task 7 — Compute coverage for company size

In [ ]:
coverage = df_us['company_size'].notna().mean() * 100
print(f"Company size coverage for US companies: {coverage:.1f}%")

**Comment:** Since only a portion of US companies have a non-null `company_size`, the size distribution in Task 6 only reflects companies that chose to report their size — conclusions may not generalise to all US companies on LinkedIn.

## Task 8 — Identify the top 10 countries

In [ ]:
top_countries = df_companies['country'].value_counts().head(10)
print("Top 10 countries by number of companies:")
print(top_countries)

**Comment:** The strong US dominance is not surprising — LinkedIn's user base and job market activity is heavily concentrated in North America, so the dataset reflects the platform's user base rather than the global job market.